In [60]:
import asyncio

import asyncio
from codecs import StreamReader
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import SelectorGroupChat

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
from autogen_agentchat.ui import Console
import os
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.http import HttpTool
from autogen_core.models import UserMessage
from autogen_ext.models.ollama import OllamaChatCompletionClient

#import from Langchain
from langchain_community.utilities import GoogleSerperAPIWrapper


In [61]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
import json


In [62]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["SERPER_API_KEY"]=os.getenv("SERPER_API_KEY")
open_router_api_key=os.getenv("OPENROUTER_API_KEY")
model=ChatGroq(model="qwen-qwq-32B")
def llm(input):
    model=ChatGroq(model="qwen-qwq-32B")
    output=model.invoke(input)
    return output.content
print(model)

model1=OpenAIChatCompletionClient(model="gpt-4o")
print(model1)

client=<groq.resources.chat.completions.Completions object at 0x12167ec60> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x121729670> model_name='qwen-qwq-32B' model_kwargs={} groq_api_key=SecretStr('**********')


In [63]:
open_router_model_client = OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="nvidia/llama-3.1-nemotron-70b-instruct",
    api_key=open_router_api_key,
    model_info={
        "family": 'deepseek',
        "vision": True,
        "function_calling": True,
        "json_output": False
    }
)
response = open_router_model_client.create([UserMessage(content="What is the capital of France?", source="user")])

print(response)

<coroutine object BaseOpenAIChatCompletionClient.create at 0x12303d700>


/Users/arunkumar/anaconda3/envs/py312/lib/python3.12/site-packages/autogen_ext/models/openai/_openai_client.py:439: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)
/var/folders/4m/z36lvyp57r37vs_gdh88c2mc0000gn/T/ipykernel_45522/2183643254.py:12: RuntimeWarning: coroutine 'BaseOpenAIChatCompletionClient.create' was never awaited
  response = open_router_model_client.create([UserMessage(content="What is the capital of France?", source="user")])


LIKE ROUNDROBIN IT HAS ANOTHER TYPE OF SELECTOR GROUP CHAT, where we can define which agent to be used in the participants for what tasks

In [64]:
#Planning agent has the control of all agents among them
planning_agent = AssistantAgent(
    name="PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model1,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

In [65]:
#TOOL DECLARATION

search_tool_wrapper = GoogleSerperAPIWrapper(type='search')

def search_web(query:str) ->str:
    """Search the web for the given query and return the results."""
    try:
        results = search_tool_wrapper.run(query)
        return results
    except Exception as e:
        print(f"Error occurred while searching the web: {e}")
        return "No results found."  

In [66]:
#JUST MANUAL WEBSEARCH TOOL
def search_web_tool(query:str)-> str:
    # Simulate a web search
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."

In [67]:
web_search_agent = AssistantAgent(
    name = 'WebSearchAgent',
    description= 'An agent for searching the web for information.',
    model_client=model1,
    #tools = [search_web_tool], # normal search
    tools = [search_web],# websearch
    reflect_on_tool_use=True,
    system_message='''
        You are a web search agent.
        Your only tool is search_web - use it to find the information you need.

        You make only one search call at a time.
        
        Once you have the results, you never do calculations or data analysis on them.
    ''',
)

In [68]:
def percentage_change_tool(start:float, end:float) -> float:
    # Calculate percentage change
    if start == 0:
        return 0
    return ((end - start) / start) * 100

In [69]:
data_analyst_agent = AssistantAgent(
    name = 'DataAnalystAgent',
    description= 'An agent for performing calculations and data analysis.',
    model_client=open_router_model_client,
    tools= [percentage_change_tool],
    system_message='''
        You are a data analyst agent.
        Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided (percentage_change_tool).

        If you have not seen the data, ask for it.

    ''',
)

Termination Condition

In [70]:

from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination

text_mention_termination = TextMentionTermination('TERMINATE')
#the total conversation between the agents should be within this range
max_message_termination = MaxMessageTermination(max_messages=10)
combined_termination = text_mention_termination | max_message_termination

In [71]:
selector_prompt = '''
Select an agent to perform the task.

{roles}

current conversation history :
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure that the planning agent has assigned task before other agents start working.
Only select one agent.
'''

In [72]:
planning_agent.description

'An agent for planning tasks, this agent should be the first to engage when given a new task.'

In [73]:
#use only open AI model for the is kind of selector else throws error model INFO
selector_team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model1,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True)

In [74]:
task = "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"


In [75]:
#intiating here:
from autogen_agentchat.ui import Console
#await Console(selector_team.run_stream(task=task))

In [76]:
# With real web search


from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To address your request, let's break it down into two main tasks:

1. Identify the Miami Heat player with the highest points in the 2006-2007 season.
2. Calculate the percentage change in total rebounds for that player between the 2007-2008 and 2008-2009 seasons.

Here's how we'll proceed:

1. WebSearchAgent: Find out the Miami Heat player with the highest points in the 2006-2007 NBA season.
2. WebSearchAgent: Gather the total rebounds for this player in both the 2007-2008 and 2008-2009 NBA seasons.
3. DataAnalystAgent: Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons for the identified player. 

Let's get started.
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[

CancelledError: 